# Module 5 – Use Case 1: Proposal Assistant (Solution (SDK))

---

## 🔧 Setup and Configuration

Let's import the shared configuration and set up our LLM helper function.

In [29]:
# Import standard libraries
import os
import json
import requests
from typing import Dict, Any, List

# Import shared configuration
from llm_config_template import (
    INFERENCE_ENDPOINT,
    API_KEY,
    MODEL_NAME,
    API_VERSION,
    validate_config
)

# Validate configuration
validate_config()
print("✅ Configuration validated successfully!")
print(f"   Using model: {MODEL_NAME}")
print(f"   Endpoint: {INFERENCE_ENDPOINT}")

✅ Configuration validated successfully!
   Endpoint: https://srika-mkndeeu4-eastus2.openai.azure.com/
   Model: gpt-4.1
   API Version: 2024-05-01-preview
✅ Configuration validated successfully!
   Using model: gpt-4.1
   Endpoint: https://srika-mkndeeu4-eastus2.openai.azure.com/


### Shared Helper Function

This helper function makes LLM calls using the Azure AI Foundry endpoint. We'll use it throughout both implementations.

In [30]:
def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.4, max_tokens: int = 1024) -> str:
    """
    Make a single LLM call using Azure AI Foundry endpoint with system and user prompts.
    
    Args:
        system_prompt: The system prompt (sets the AI's behavior/role)
        user_prompt: The user prompt (the actual request/query)
        temperature: Controls randomness (0.0 = deterministic, 1.0 = creative)
        max_tokens: Maximum length of the response
    
    Returns:
        The model's response as a string
    """
    url = f"https://srika-mkndeeu4-eastus2.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": API_KEY,
        "x-ms-model-mesh-model-name": MODEL_NAME,
    }
    
    body = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    
    response = requests.post(url, headers=headers, json=body)
    response.raise_for_status()
    data = response.json()
    
    return data["choices"][0]["message"]["content"]


def call_llm_with_messages(messages: List[Dict[str, str]], temperature: float = 0.4, max_tokens: int = 1024) -> str:
    """
    Make an LLM call with a full message history (useful for agentic patterns).
    
    Args:
        messages: List of message dicts with 'role' and 'content'
        temperature: Controls randomness
        max_tokens: Maximum length of the response
    
    Returns:
        The model's response as a string
    """
    url = f"{INFERENCE_ENDPOINT}/chat/completions?api-version={API_VERSION}"
    
    headers = {
        "Content-Type": "application/json",
        "api-key": API_KEY,
        "x-ms-model-mesh-model-name": MODEL_NAME,
    }
    
    body = {
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    
    response = requests.post(url, headers=headers, json=body)
    response.raise_for_status()
    data = response.json()
    
    return data["choices"][0]["message"]["content"]


print("✅ Helper functions defined successfully!")

✅ Helper functions defined successfully!


---

## 📘 Quick Reminder: Proposal Assistant Use Case

### The Business Problem

In the workspace notebook (`M5_UC1_Proposal_Workspace.ipynb`), you explored the **Proposal Assistant** use case:

* **Input**: A high-level requirement or RFP-style description about a GenAI/Azure solution
* **Output**: A structured, client-ready proposal document
* **Challenge**: Manual proposal writing is time-consuming (4-8 hours), quality varies, and needs balance technical depth with business value

### Typical Proposal Structure

A complete proposal includes these sections:

1. **Executive Summary / Introduction**
2. **Business Context & Problem Statement**
3. **Proposed GenAI Solution Overview**
4. **Implementation Approach** (phases, milestones, timeline)
5. **Effort Estimation & Resource Requirements**
6. **Risks, Assumptions & Mitigation Strategies**
7. **Next Steps & Call to Action**

### What You Did in the Workspace

In the workspace notebook, you:

* ✅ Explored the business problem and requirements
* ✅ Chose an architecture approach (prompt app / workflow / agent / hybrid)
* ✅ Designed inputs, outputs, steps/roles, and evaluation criteria
* ✅ (Optional) Started sketching implementation code

Now you'll see **two complete implementations** using only the OpenAI/Azure SDK.

### Sample Requirement

We'll use this sample requirement throughout both implementations:

---

**📋 Sample Requirement: Document Intelligence Assistant**

**Client / Business Unit:** Legal Operations Department, ABC Financial Services

**Domain / Industry:** Financial Services / Banking

**Problem Statement:**
Our legal team processes hundreds of contracts, compliance documents, and regulatory filings each month. Currently, lawyers spend 40-60% of their time on manual document review: extracting key clauses, identifying risks, checking compliance, and summarizing findings for stakeholders. This is unsustainable as document volume grows 20% year-over-year.

**Desired Outcomes:**
- Reduce document review time by 50-70%
- Improve consistency in risk identification across reviewers
- Enable faster response to regulatory queries
- Free up lawyers to focus on high-value strategic work
- Maintain or improve accuracy (currently 95% accuracy on manual review)

**Constraints:**
- **Budget:** $200K-$300K for MVP (6-month project)
- **Technology:** Must use Azure (existing enterprise agreement), prefer Azure OpenAI Service
- **Compliance:** Subject to financial services regulations (SOC 2, data residency requirements)
- **Timeline:** Need MVP in production within 6 months

**Non-Functional Requirements:**
- **Performance:** Process a 50-page contract in under 2 minutes
- **Security:** End-to-end encryption, role-based access control, audit logging
- **Scalability:** Handle 500 documents per day initially, scale to 2000/day within 12 months
- **Integration:** Must integrate with existing document management system (SharePoint) and case management tool

---

In [31]:
# Store the sample requirement as a variable we'll use throughout
SAMPLE_REQUIREMENT = """
**Client / Business Unit:** Legal Operations Department, ABC Financial Services

**Domain / Industry:** Financial Services / Banking

**Problem Statement:**
Our legal team processes hundreds of contracts, compliance documents, and regulatory filings each month. Currently, lawyers spend 40-60% of their time on manual document review: extracting key clauses, identifying risks, checking compliance, and summarizing findings for stakeholders. This is unsustainable as document volume grows 20% year-over-year.

**Desired Outcomes:**
- Reduce document review time by 50-70%
- Improve consistency in risk identification across reviewers
- Enable faster response to regulatory queries
- Free up lawyers to focus on high-value strategic work
- Maintain or improve accuracy (currently 95% accuracy on manual review)

**Constraints:**
- Budget: $200K-$300K for MVP (6-month project)
- Technology: Must use Azure (existing enterprise agreement), prefer Azure OpenAI Service
- Compliance: Subject to financial services regulations (SOC 2, data residency requirements)
- Timeline: Need MVP in production within 6 months

**Non-Functional Requirements:**
- Performance: Process a 50-page contract in under 2 minutes
- Security: End-to-end encryption, role-based access control, audit logging
- Scalability: Handle 500 documents per day initially, scale to 2000/day within 12 months
- Integration: Must integrate with existing document management system (SharePoint) and case management tool
"""

print("✅ Sample requirement loaded!")

✅ Sample requirement loaded!


---

## Section 3: Deterministic Workflow Solution

### 3.1 Workflow Architecture

**Conceptual Flow:**

```
                      Start
                        |
                        v
              +------------------+
              | Generate Outline |
              +------------------+
                        |
                        v
         +----------------------------+
         | Generate All Sections      |
         | (one by one, sequentially) |
         +----------------------------+
                        |
                        v
             +--------------------+
             | Refine Style       |
             | (polish language)  |
             +--------------------+
                        |
                        v
                  Final Proposal
```

**Key Characteristics:**
- **Fixed sequence** of steps
- **No branching** or dynamic decisions based on output quality
- **No iteration** — each step runs exactly once
- **Predictable** execution path

This is a classic **workflow** (or "chain" or "pipeline"). Each function is a discrete step that transforms input into output, and the steps are composed in a deterministic order.

### 3.2 Core Workflow Functions

We'll define four functions that compose into our workflow:

1. **`generate_outline()`** — Given a requirement, produce a structured outline for the proposal.
2. **`generate_section()`** — Given a requirement, an outline, and a section name, generate the content for that section.
3. **`generate_all_sections()`** — Orchestrate section generation for all sections in the outline.
4. **`refine_style()`** — Polish the final proposal for tone, clarity, and professional language.

In [32]:
def generate_outline(requirement_text: str) -> str:
    """
    Generate a structured outline for a proposal based on the requirement.
    
    Args:
        requirement_text: The business requirement as a string
        
    Returns:
        A structured outline as a string with sections and subsections
    """
    system_prompt = """You are a proposal expert. Your job is to create a clear, structured outline for a technical proposal based on a client requirement.

The outline should include standard sections like:
- Executive Summary
- Problem Statement
- Proposed Solution
- Technical Architecture
- Implementation Plan
- Timeline & Milestones
- Budget & Resources
- Risk & Mitigation
- Success Metrics

Output the outline in a structured format with section names and brief descriptions of what each section should cover."""
    
    user_prompt = f"""Based on this requirement, create a detailed proposal outline:

{requirement_text}

Provide a structured outline with section names and what each section should address."""
    
    outline = call_llm(system_prompt, user_prompt)
    return outline

print("✅ generate_outline() defined")

✅ generate_outline() defined


In [33]:
def generate_section(requirement_text: str, outline: str, section_name: str) -> str:
    """
    Generate content for a specific section of the proposal.
    
    Args:
        requirement_text: The original business requirement
        outline: The full proposal outline for context
        section_name: The name of the section to generate
        
    Returns:
        Generated content for the specified section
    """
    system_prompt = """You are a proposal writer. You will be given:
1. A client requirement
2. An overall proposal outline
3. A specific section name to write

Generate detailed, professional content for that section. Be specific, use technical language where appropriate, and address the client's needs directly. Write 2-4 paragraphs for the section."""
    
    user_prompt = f"""Client Requirement:
{requirement_text}

Proposal Outline:
{outline}

Please write the content for the section: **{section_name}**

Provide detailed, professional content that fits within the overall proposal structure."""
    
    section_content = call_llm(system_prompt, user_prompt)
    return section_content

print("✅ generate_section() defined")

✅ generate_section() defined


In [34]:
from typing import Dict

def generate_all_sections(requirement_text: str, outline: str) -> Dict[str, str]:
    """
    Generate content for all sections in the outline.
    
    Args:
        requirement_text: The original business requirement
        outline: The full proposal outline
        
    Returns:
        Dictionary mapping section names to their generated content
    """
    # For simplicity, we'll define the key sections we want to generate
    # In a real system, we might parse the outline to extract section names
    sections = [
        "Executive Summary",
        "Problem Statement",
        "Proposed Solution",
        "Technical Architecture",
        "Implementation Plan",
        "Timeline & Milestones",
        "Budget & Resources",
        "Success Metrics"
    ]
    
    sections_content = {}
    
    for section_name in sections:
        print(f"  → Generating: {section_name}...")
        content = generate_section(requirement_text, outline, section_name)
        sections_content[section_name] = content
    
    return sections_content

print("✅ generate_all_sections() defined")

✅ generate_all_sections() defined


In [35]:
def refine_style(full_text: str, style_guidance: str = "professional, clear, client-focused") -> str:
    """
    Polish the proposal for tone, clarity, and style.
    
    Args:
        full_text: The complete proposal text
        style_guidance: Description of desired style
        
    Returns:
        Refined proposal text
    """
    system_prompt = """You are an editor specializing in business proposals. Your job is to refine and polish proposal text to be clear, professional, and client-focused.

Improve:
- Clarity and readability
- Professional tone
- Consistency in terminology
- Flow between sections

Do NOT change the technical content or facts. Only improve the writing quality."""
    
    user_prompt = f"""Please refine this proposal to be {style_guidance}.

Proposal:
{full_text}

Return the refined version."""
    
    refined = call_llm(system_prompt, user_prompt)
    return refined

print("✅ refine_style() defined")

✅ refine_style() defined


### 3.3 Workflow Driver Function

Now we compose these functions into a single workflow that executes sequentially:

In [36]:
def build_proposal_workflow(requirement_text: str, style: str = "professional, clear, client-focused") -> Dict[str, any]:
    """
    Execute the complete proposal generation workflow.
    
    This is a DETERMINISTIC WORKFLOW:
    - Fixed sequence of steps
    - No branching or iteration
    - Each step runs exactly once
    
    Args:
        requirement_text: The business requirement
        style: Desired writing style for refinement
        
    Returns:
        Dictionary containing outline, sections, final proposal, and metadata
    """
    print("🚀 Starting Proposal Generation Workflow\n")
    
    # Step 1: Generate outline
    print("Step 1: Generating outline...")
    outline = generate_outline(requirement_text)
    print("  ✅ Outline complete\n")
    
    # Step 2: Generate all sections
    print("Step 2: Generating sections...")
    sections = generate_all_sections(requirement_text, outline)
    print("  ✅ All sections complete\n")
    
    # Step 3: Assemble full proposal
    print("Step 3: Assembling proposal...")
    full_proposal = "\n\n".join([f"## {name}\n\n{content}" for name, content in sections.items()])
    print("  ✅ Proposal assembled\n")
    
    # Step 4: Refine style
    print("Step 4: Refining style...")
    final_proposal = refine_style(full_proposal, style)
    print("  ✅ Style refinement complete\n")
    
    print("✅ Workflow complete!")
    
    return {
        "outline": outline,
        "sections": sections,
        "final_proposal": final_proposal,
        "metadata": {
            "approach": "deterministic_workflow",
            "steps": ["generate_outline", "generate_sections", "assemble", "refine_style"],
            "num_sections": len(sections)
        }
    }

print("✅ build_proposal_workflow() defined")

✅ build_proposal_workflow() defined


**Run the workflow:**

In [37]:
# Run the workflow
result = build_proposal_workflow(SAMPLE_REQUIREMENT)

🚀 Starting Proposal Generation Workflow

Step 1: Generating outline...
  ✅ Outline complete

Step 2: Generating sections...
  → Generating: Executive Summary...
  → Generating: Problem Statement...
  → Generating: Proposed Solution...
  → Generating: Technical Architecture...
  → Generating: Implementation Plan...
  → Generating: Timeline & Milestones...
  → Generating: Budget & Resources...
  → Generating: Success Metrics...
  ✅ All sections complete

Step 3: Assembling proposal...
  ✅ Proposal assembled

Step 4: Refining style...
  ✅ Style refinement complete

✅ Workflow complete!


In [38]:
# Display results
print("\n" + "="*60)
# print("FINAL PROPOSAL (First 1000 characters):")
print("FINAL PROPOSAL:")
print("="*60)
# print(result["final_proposal"][:1000] + "...")
print(result["final_proposal"])


FINAL PROPOSAL:
Certainly! Here is your refined, professional, and client-focused proposal:

---

## Executive Summary

ABC Financial Services’ Legal Operations Department is at a pivotal juncture as the volume and complexity of contracts, compliance documents, and regulatory filings increase by 20% annually. Currently, highly skilled legal professionals spend 40–60% of their time on manual document review tasks—including clause extraction, risk identification, compliance verification, and stakeholder reporting. This manual approach is unsustainable, leading to operational inefficiencies, delayed regulatory responses, and inconsistencies in risk assessment.

To address these challenges, we propose implementing an AI-powered document review and risk identification platform, purpose-built on Microsoft Azure and leveraging Azure OpenAI Service. This solution will automate key aspects of the document review lifecycle, including intelligent clause extraction, advanced risk identification, 

### 3.4 Discussion: Why Is This a Workflow?

**Characteristics of a Workflow/Chain:**

✅ **Fixed sequence**: The steps always execute in the same order:  
   `outline → sections → assemble → refine`

✅ **No dynamic branching**: The control flow doesn't change based on intermediate results. We don't check if the outline is good enough and decide to skip section generation.

✅ **No iteration**: Each step runs exactly once. We don't loop back to regenerate sections if they're low quality.

✅ **Predictable**: Given the same input, the execution path is always the same (though the LLM outputs may vary).

**Pros:**
- **Simple to understand and debug** — linear execution makes it easy to trace
- **Fast to implement** — no complex orchestration logic needed
- **Predictable behavior** — always runs the same steps
- **Easy to test** — each function can be tested in isolation

**Cons:**
- **No quality control** — if a section is poorly generated, we don't detect it or retry
- **Inflexible** — can't adapt to different requirement types or handle errors gracefully
- **Potentially wasteful** — always runs all steps even if some might not be needed
- **No learning** — doesn't improve based on feedback or previous results

**When to use workflows:**
- When the problem is well-defined and the steps are clear
- When quality is acceptable without iteration
- When speed and simplicity are more important than perfection
- When you want predictable, reproducible behavior

---

## Section 4: Agentic Solution

### 4.1 Agentic Architecture

**Conceptual Flow:**

```
                          Start
                            |
                            v
                   +----------------+
                   | Planner Agent  |
                   | (creates plan) |
                   +----------------+
                            |
                            v
                   +----------------+
                   | Drafter Agent  |
                   | (writes draft) |
                   +----------------+
                            |
                            v
                   +------------------+
                   | Evaluator Agent  |
                   | (checks quality) |
                   +------------------+
                            |
                    /---------------\
                   /  Quality OK?    \
                  /                   \
                 /                     \
                NO                      YES
                |                        |
                v                        v
         +-----------+            Final Proposal
         | Iterate:  |
         | - Refine  |
         | - Redraft |
         +-----------+
                |
                v
         (back to Drafter)
```

**Key Characteristics:**
- **Dynamic control flow** — execution path depends on evaluator's assessment
- **Iteration loop** — can regenerate content multiple times until quality threshold is met
- **Agent roles** — each agent has a specific responsibility and expertise
- **Goal-directed** — the system works toward a quality goal, not just executing steps

This is a classic **agentic pattern**. The orchestrator dynamically decides what to do next based on intermediate results.

### 4.2 Agent Functions

We'll define three specialized agents, each with a distinct role:

In [39]:
def planner_agent(requirement_text: str) -> Dict[str, any]:
    """
    PLANNER AGENT: Analyzes the requirement and creates a strategic plan.
    
    Role: Think strategically about what the proposal needs to accomplish.
    
    Args:
        requirement_text: The business requirement
        
    Returns:
        Dictionary containing plan structure and key points to address
    """
    system_prompt = """You are a strategic planner for technical proposals. Your role is to:

1. Analyze the client's requirement deeply
2. Identify key concerns, constraints, and success criteria
3. Determine what sections the proposal must include
4. Define the core messages for each section
5. Highlight risks or gaps that must be addressed

Output a structured plan with:
- List of proposal sections
- Key points each section must address
- Client hot buttons (what they care most about)
- Potential risks to mitigate"""
    
    user_prompt = f"""Analyze this requirement and create a strategic plan for the proposal:

{requirement_text}

Provide a detailed plan for how to approach this proposal."""
    
    plan_text = call_llm(system_prompt, user_prompt)
    
    return {
        "plan": plan_text,
        "agent": "planner"
    }

print("✅ planner_agent() defined")

✅ planner_agent() defined


In [40]:
def drafter_agent(requirement_text: str, plan: str, feedback: str = None) -> Dict[str, any]:
    """
    DRAFTER AGENT: Writes the proposal based on the plan and any feedback.
    
    Role: Execute the plan by writing clear, compelling proposal content.
    
    Args:
        requirement_text: The business requirement
        plan: The strategic plan from the planner agent
        feedback: Optional feedback from evaluator to incorporate
        
    Returns:
        Dictionary containing the draft proposal
    """
    system_prompt = """You are a proposal writer. Your role is to:

1. Follow the strategic plan provided
2. Write clear, professional, client-focused content
3. Address all key points and concerns
4. Use specific details from the requirement
5. If feedback is provided, incorporate it to improve the draft

Write a complete proposal with all sections. Be thorough and professional."""
    
    user_prompt = f"""Client Requirement:
{requirement_text}

Strategic Plan:
{plan}
"""
    
    if feedback:
        user_prompt += f"""

Previous Feedback (incorporate this):
{feedback}
"""
    
    user_prompt += "\n\nWrite a complete proposal following the plan."
    
    draft = call_llm(system_prompt, user_prompt)
    
    return {
        "draft": draft,
        "agent": "drafter",
        "incorporated_feedback": feedback is not None
    }

print("✅ drafter_agent() defined")

✅ drafter_agent() defined


In [41]:
def evaluator_agent(requirement_text: str, draft: str, plan: str) -> Dict[str, any]:
    """
    EVALUATOR AGENT: Assesses the quality of the draft and provides feedback.
    
    Role: Act as a quality gate — determine if the proposal is ready or needs improvement.
    
    Args:
        requirement_text: The original requirement
        draft: The draft proposal to evaluate
        plan: The strategic plan to check against
        
    Returns:
        Dictionary containing quality score (0-10), approved status, and feedback
    """
    system_prompt = """You are a proposal quality evaluator. Your role is to:

1. Assess if the draft addresses all requirements
2. Check if it follows the strategic plan
3. Evaluate clarity, professionalism, and persuasiveness
4. Identify gaps, weaknesses, or areas for improvement
5. Provide a quality score (0-10) and specific feedback

Output format:
- Score: [0-10]
- Approved: [YES/NO] (YES if score >= 7)
- Feedback: [Specific, actionable feedback for improvement]

Be critical but constructive. If the draft is strong, approve it. If not, provide clear guidance on what needs to change."""
    
    user_prompt = f"""Client Requirement:
{requirement_text}

Strategic Plan:
{plan}

Draft Proposal:
{draft}

Evaluate this draft. Provide a score (0-10), approval decision (YES/NO), and specific feedback."""
    
    evaluation = call_llm(system_prompt, user_prompt)
    
    # Parse the evaluation (simple heuristic)
    # In production, you'd use structured output or more robust parsing
    score = 0
    approved = False
    feedback = evaluation
    
    # Try to extract score
    import re
    score_match = re.search(r'[Ss]core:\s*(\d+)', evaluation)
    if score_match:
        score = int(score_match.group(1))
    
    # Try to extract approval
    if re.search(r'Approved:\s*YES', evaluation, re.IGNORECASE):
        approved = True
    elif re.search(r'Approved:\s*NO', evaluation, re.IGNORECASE):
        approved = False
    else:
        # Fallback: if score >= 7, approve
        approved = score >= 7
    
    return {
        "score": score,
        "approved": approved,
        "feedback": feedback,
        "agent": "evaluator"
    }

print("✅ evaluator_agent() defined")

✅ evaluator_agent() defined


### 4.3 Agentic Orchestrator

Now we implement the orchestrator that coordinates the agents in an iterative loop:

In [42]:
def run_agentic_pipeline(requirement_text: str, max_iterations: int = 3) -> Dict[str, any]:
    """
    Execute the agentic proposal generation pipeline.
    
    This is an AGENTIC PATTERN:
    - Dynamic control flow based on evaluation results
    - Iterative refinement until quality threshold is met
    - Agents collaborate with specific roles
    - Goal-directed behavior (achieve quality score >= 7)
    
    Args:
        requirement_text: The business requirement
        max_iterations: Maximum number of draft-evaluate cycles
        
    Returns:
        Dictionary containing final proposal, iteration log, and metadata
    """
    print("🤖 Starting Agentic Proposal Generation Pipeline\n")
    
    # Step 1: Planner creates strategic plan
    print("Step 1: Planner Agent - Creating strategic plan...")
    plan_result = planner_agent(requirement_text)
    plan = plan_result["plan"]
    print("  ✅ Plan created\n")
    
    # Iteration loop: Draft → Evaluate → Refine
    iteration_log = []
    final_draft = None
    feedback = None
    
    for iteration in range(1, max_iterations + 1):
        print(f"--- Iteration {iteration} ---")
        
        # Step 2: Drafter creates/revises proposal
        print(f"  Drafter Agent - {'Creating initial draft' if iteration == 1 else 'Revising based on feedback'}...")
        draft_result = drafter_agent(requirement_text, plan, feedback)
        draft = draft_result["draft"]
        print("    ✅ Draft ready")
        
        # Step 3: Evaluator assesses quality
        print("  Evaluator Agent - Assessing quality...")
        eval_result = evaluator_agent(requirement_text, draft, plan)
        score = eval_result["score"]
        approved = eval_result["approved"]
        feedback = eval_result["feedback"]
        
        print(f"    📊 Score: {score}/10")
        print(f"    {'✅ APPROVED' if approved else '❌ NEEDS IMPROVEMENT'}")
        
        # Log this iteration
        iteration_log.append({
            "iteration": iteration,
            "score": score,
            "approved": approved,
            "draft_length": len(draft)
        })
        
        # Decision point: approve or iterate?
        if approved:
            print(f"\n✅ Proposal approved after {iteration} iteration(s)!")
            final_draft = draft
            break
        else:
            print("  🔄 Sending feedback to drafter for revision...\n")
            if iteration == max_iterations:
                print(f"⚠️  Reached max iterations ({max_iterations}). Using last draft.")
                final_draft = draft
    
    print("\n✅ Agentic pipeline complete!")
    
    return {
        "final_proposal": final_draft,
        "plan": plan,
        "iterations": iteration_log,
        "total_iterations": len(iteration_log),
        "metadata": {
            "approach": "agentic",
            "max_iterations": max_iterations,
            "agents": ["planner", "drafter", "evaluator"]
        }
    }

print("✅ run_agentic_pipeline() defined")

✅ run_agentic_pipeline() defined


### 4.4 Run Agentic Demo

In [43]:
# Run the agentic pipeline
agentic_result = run_agentic_pipeline(SAMPLE_REQUIREMENT, max_iterations=3)

# Display results
print("\n" + "="*60)
print("ITERATION LOG:")
print("="*60)
for log in agentic_result["iterations"]:
    print(f"Iteration {log['iteration']}: Score {log['score']}/10 - {'APPROVED ✅' if log['approved'] else 'NEEDS WORK ❌'}")

print("\n" + "="*60)
print("FINAL PROPOSAL (First 1000 characters):")
print("="*60)
print(agentic_result["final_proposal"][:1000] + "...")

🤖 Starting Agentic Proposal Generation Pipeline

Step 1: Planner Agent - Creating strategic plan...
  ✅ Plan created

--- Iteration 1 ---
  Drafter Agent - Creating initial draft...
    ✅ Draft ready
  Evaluator Agent - Assessing quality...
    📊 Score: 0/10
    ❌ NEEDS IMPROVEMENT
  🔄 Sending feedback to drafter for revision...

--- Iteration 2 ---
  Drafter Agent - Revising based on feedback...
    ✅ Draft ready
  Evaluator Agent - Assessing quality...
    📊 Score: 0/10
    ❌ NEEDS IMPROVEMENT
  🔄 Sending feedback to drafter for revision...

--- Iteration 3 ---
  Drafter Agent - Revising based on feedback...
    ✅ Draft ready
  Evaluator Agent - Assessing quality...
    📊 Score: 0/10
    ❌ NEEDS IMPROVEMENT
  🔄 Sending feedback to drafter for revision...

⚠️  Reached max iterations (3). Using last draft.

✅ Agentic pipeline complete!

ITERATION LOG:
Iteration 1: Score 0/10 - NEEDS WORK ❌
Iteration 2: Score 0/10 - NEEDS WORK ❌
Iteration 3: Score 0/10 - NEEDS WORK ❌

FINAL PROPOSAL (Fi

### 4.5 Discussion: Why Is This Agentic?

**Characteristics of an Agentic Pattern:**

✅ **Dynamic control flow**: The execution path depends on the evaluator's assessment. If the draft is approved, we stop. If not, we iterate.

✅ **Iteration and refinement**: The drafter can run multiple times, incorporating feedback each time. The system learns from previous attempts.

✅ **Goal-directed behavior**: The system has a clear goal (quality score >= 7) and works toward achieving it. It's not just executing steps blindly.

✅ **Agent roles and collaboration**: Each agent has specialized expertise (planning, drafting, evaluation) and they collaborate to achieve the goal.

✅ **Adaptive**: The system can handle different scenarios — if the first draft is perfect, it stops early. If quality is low, it iterates up to the max limit.

**Pros:**
- **Quality control** — built-in evaluation ensures output meets standards
- **Self-improving** — can iterate until reaching acceptable quality
- **Flexible** — adapts to different requirement complexities
- **Role specialization** — each agent focuses on what it does best
- **Transparent** — iteration logs provide insight into the process

**Cons:**
- **More complex** — harder to implement and debug than a simple workflow
- **Unpredictable execution time** — could take 1 iteration or max iterations
- **Higher cost** — more LLM calls due to iteration
- **Potential infinite loops** — need safeguards (max iterations) to prevent runaway execution
- **Harder to test** — non-deterministic behavior makes testing more challenging

**When to use agents:**
- When quality is critical and worth the extra cost
- When the problem requires iteration and refinement
- When you need adaptability to handle diverse inputs
- When you can afford unpredictable execution time
- When you want built-in quality assurance

---

## Section 5: Workflow vs. Agent — When to Choose?

### Comparison Table

| **Dimension**          | **Workflow (Deterministic Chain)** | **Agent (Iterative with Evaluation)** |
|------------------------|-------------------------------------|----------------------------------------|
| **Control Flow**       | Fixed sequence                      | Dynamic, based on evaluation           |
| **Iteration**          | None (each step runs once)          | Multiple iterations possible           |
| **Quality Assurance**  | None (assumes good output)          | Built-in evaluation and refinement     |
| **Predictability**     | High (same steps every time)        | Lower (depends on evaluation)          |
| **Complexity**         | Low (simple to implement)           | Higher (orchestration + evaluation)    |
| **Cost**               | Lower (fewer LLM calls)             | Higher (multiple iterations)           |
| **Speed**              | Faster (no iteration)               | Slower (iteration overhead)            |
| **Adaptability**       | Low (fixed path)                    | High (adapts to quality feedback)      |
| **Debugging**          | Easier (linear flow)                | Harder (non-deterministic)             |
| **Use Cases**          | Simple, well-defined tasks          | Complex, quality-critical tasks        |

### When to Choose a Workflow:

✅ **The problem is well-defined** with clear steps  
✅ **Speed matters** more than perfection  
✅ **Cost is a concern** (minimize LLM calls)  
✅ **Predictability is important** (same execution path every time)  
✅ **The output is "good enough"** without iteration  

**Examples:**
- Generating a simple email from a template
- Creating a standard report with fixed sections
- Summarizing a document with predefined structure
- Basic data transformation or formatting tasks

### When to Choose an Agent:

✅ **Quality is critical** and worth the extra cost  
✅ **The task requires refinement** or iteration  
✅ **Diverse inputs** require adaptive handling  
✅ **Built-in quality checks** are needed  
✅ **You can tolerate unpredictable execution time**  

**Examples:**
- Writing high-stakes proposals, contracts, or reports
- Code generation with quality validation
- Complex problem-solving that requires multiple attempts
- Creative tasks where iteration improves output
- Tasks with quality thresholds (e.g., "score must be >= 8")

### Hybrid Approaches:

In practice, you might combine both patterns:

- **Workflow with agent nodes**: Use a workflow structure, but have specific nodes be agents that can iterate
- **Multi-agent workflows**: Multiple agents work together in a fixed sequence
- **Conditional branching**: Start with a workflow, but add decision points that can trigger iteration

**Key Takeaway:**  
Choose the pattern that matches your **quality requirements**, **cost constraints**, and **complexity tolerance**. Start simple (workflow) and add agency (iteration, evaluation) only where needed.

---

## Section 6: Adding RAG & External Knowledge

Both the workflow and agentic patterns can be enhanced with **Retrieval-Augmented Generation (RAG)** to ground responses in external knowledge.

### What is RAG?

**RAG** = Retrieval + Generation
1. **Retrieve** relevant documents/data from a knowledge base
2. **Augment** the LLM prompt with retrieved context
3. **Generate** a response grounded in that context

This is crucial for:
- **Factual accuracy** (grounding in real documents rather than hallucinating)
- **Domain expertise** (using company-specific knowledge)
- **Compliance** (citing sources and maintaining auditability)

### RAG Integration Points in Our Patterns

#### In the Workflow Pattern:

```python
# BEFORE: generate_outline(requirement_text)
# AFTER: Add retrieval step

def generate_outline_with_rag(requirement_text: str) -> str:
    # 1. Retrieve relevant past proposals, templates, domain guidelines
    retrieved_docs = retrieve_similar_proposals(requirement_text)
    
    # 2. Augment the prompt with retrieved context
    context = "\n".join([f"Reference {i+1}:\n{doc}" for i, doc in enumerate(retrieved_docs)])
    
    system_prompt = f"""You are a proposal expert. Use these reference documents to inform your outline:

{context}

Now create an outline for the new requirement..."""
    
    # 3. Generate with augmented context
    outline = call_llm(system_prompt, requirement_text)
    return outline
```

You could add RAG at any step:
- **Generate outline**: Retrieve similar proposal structures
- **Generate sections**: Retrieve domain-specific content (technical specs, case studies)
- **Refine style**: Retrieve company style guides

#### In the Agentic Pattern:

```python
# Add RAG to the planner agent
def planner_agent_with_rag(requirement_text: str) -> Dict[str, any]:
    # Retrieve relevant strategic plans, competitive analyses
    retrieved_strategy = retrieve_strategic_docs(requirement_text)
    
    system_prompt = f"""You are a strategic planner. Use these references:

{retrieved_strategy}

Now analyze the requirement and create a strategic plan..."""
    
    plan = call_llm(system_prompt, requirement_text)
    return {"plan": plan, "agent": "planner"}
```

**Key Insight:**  
RAG is **orthogonal** to the workflow/agent choice. You can add retrieval to either pattern wherever you need external knowledge.

### Tools for RAG

- **Azure AI Search**: Vector search over documents
- **Azure Cognitive Search**: Hybrid search (keyword + semantic)
- **LangChain Retrievers**: Abstractions for different retrieval strategies
- **Custom vector databases**: Pinecone, Weaviate, Chroma, FAISS

### Next Steps for RAG

To add RAG to this use case, you would:
1. **Index knowledge**: Past proposals, templates, domain docs → vector database
2. **Implement retrieval**: Query the index with the requirement text
3. **Augment prompts**: Add retrieved context to system/user prompts
4. **Cite sources**: Track which documents were used for auditability

**Note:** We're not implementing full RAG here to keep the focus on workflow vs. agent patterns. But it's a natural next step for production systems.

---

## Section 7: Moving to LangChain

### Why LangChain?

In this notebook, we've built workflows and agents using the **pure OpenAI SDK** to understand the fundamental patterns. In production, you'll likely want to use a **framework like LangChain** that provides:

✅ **Higher-level abstractions**: `Chain`, `Agent`, `Tool` classes instead of manual orchestration  
✅ **Built-in patterns**: Pre-built chains (sequential, map-reduce, router) and agent types (ReAct, structured chat)  
✅ **Tool integration**: Easy integration with external tools, APIs, and databases  
✅ **Memory management**: Built-in conversation memory and state management  
✅ **Debugging & observability**: LangSmith for tracing and monitoring  
✅ **Community ecosystem**: Hundreds of integrations and pre-built components  

### Converting This Use Case to LangChain

**Workflow Pattern → LangChain Chain:**
```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import AzureChatOpenAI

# Create LLM
llm = AzureChatOpenAI(...)

# Create chains for each step
outline_chain = ChatPromptTemplate(...) | llm | StrOutputParser()
section_chain = ChatPromptTemplate(...) | llm | StrOutputParser()
refine_chain = ChatPromptTemplate(...) | llm | StrOutputParser()

# Compose into workflow
def build_proposal_langchain(requirement):
    outline = outline_chain.invoke({"requirement": requirement})
    sections = [section_chain.invoke({"requirement": requirement, "outline": outline, "section": s}) for s in section_names]
    final = refine_chain.invoke({"proposal": "\\n\\n".join(sections)})
    return final
```

**Agentic Pattern → LangChain Agent:**
```python
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import tool

# Define tools
@tool
def plan_proposal(requirement: str) -> str:
    \"\"\"Create strategic plan for proposal\"\"\"
    return planner_agent(requirement)

@tool
def draft_proposal(requirement: str, plan: str) -> str:
    \"\"\"Write proposal draft\"\"\"
    return drafter_agent(requirement, plan)

# Create agent
tools = [plan_proposal, draft_proposal]
agent = create_react_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools)

# Run agent
result = executor.invoke({"input": requirement})
```

### LangChain Benefits for This Use Case

- **Less boilerplate**: No need to manually construct HTTP requests
- **Built-in retry/error handling**: Automatic retries and fallbacks
- **Streaming support**: Stream tokens as they're generated
- **Async execution**: Parallel execution of independent steps
- **Prompt management**: Version and track prompts separately from code
- **LangSmith integration**: Visualize execution traces and debug issues

### Your Homework: LangChain Conversion

In a separate notebook (`M5_UC1_LangChain_Conversion_Template.ipynb`), we provide a template for converting these implementations to LangChain. You'll:

1. **Set up LangChain** with Azure OpenAI integration
2. **Convert the workflow** to use `LangChain Chains`
3. **Convert the agent** to use `LangChain Agents` with tools
4. **Compare**: Measure performance, cost, and code complexity

This will give you hands-on experience with both pure SDK and framework approaches, helping you choose the right tool for your projects.

### Key Takeaway

**Start with fundamentals** (pure SDK) to understand the patterns.  
**Move to frameworks** (LangChain) for production to gain productivity and robustness.  
**Know both** so you can choose the right level of abstraction for each project.

---

## Section 8: Reflection & Next Steps

### What We've Built

In this notebook, you've seen two complete implementations of a proposal generation system:

✅ **Workflow Pattern** (Section 3):
- Fixed sequence: outline → sections → assemble → refine
- Simple, predictable, fast
- Good for well-defined problems

✅ **Agentic Pattern** (Section 4):
- Dynamic flow: plan → draft → evaluate → iterate
- Quality-focused, adaptive, self-improving
- Good for complex, quality-critical tasks

Both patterns use the **pure OpenAI SDK** to show you what's happening under the hood, without framework magic.

### Key Takeaways

1. **Pattern choice matters**: Match the pattern to your requirements (quality vs. speed, cost vs. adaptability)

2. **Workflows are underrated**: Don't over-engineer. If a simple chain works, use it.

3. **Agents add intelligence**: When quality matters, agents with evaluation loops are worth the complexity.

4. **RAG is orthogonal**: You can add retrieval to either pattern for knowledge grounding.

5. **Frameworks help production**: LangChain/LangGraph provide abstractions that reduce boilerplate and increase reliability.

### Reflection Questions

Before moving on, take a moment to reflect:

1. **For your current projects**, which pattern would be most appropriate? Why?

2. **What trade-offs** are most important in your context? (Speed, cost, quality, predictability?)

3. **Where would you add RAG** in these implementations? What knowledge sources would you use?

4. **How would you test** these systems? What metrics would you track?

5. **What happens when the LLM fails** or produces low-quality output? How would you handle errors?

### Next Steps

📘 **Continue learning**:
- Explore `M5_UC1_LangChain_Conversion_Template.ipynb` to see these patterns in LangChain
- Experiment with different agent configurations (more agents, different roles)
- Try different evaluation criteria (readability, compliance, creativity)

🛠️ **Build your own**:
- Adapt this use case to your domain (code generation, data analysis, content creation)
- Add RAG integration with your organization's knowledge base
- Implement monitoring and logging for production use

🎯 **Master the fundamentals**:
- Understand when to use workflows vs. agents
- Learn to compose complex systems from simple patterns
- Build intuition for LLM orchestration

### Final Thought

> **"The best architecture is the simplest one that meets your requirements."**

Start with a workflow. Add agency when you need it. Always measure the trade-offs.

---

**Congratulations!** You now understand the core patterns for building production LLM systems. 🎉